# Thai Sign Language — Model Evaluation on Test Videos

> Loads the trained LSTM model from `models/` and evaluates it on every `.mp4` file in `test_data/`.  
> Features are extracted using the **same MediaPipe pipeline** as preprocessing so results are directly comparable.

| Cell | Purpose |
|------|---------|
| 1 | Imports & configuration |
| 2 | Load model & class metadata |
| 3 | Feature-extraction pipeline (MediaPipe → keypoints → features → normalize) |
| 4 | Scan `test_data/` and predict every video |
| 5 | Overall accuracy + classification report |
| 6 | Confusion matrix heatmap |
| 7 | Confidence score distribution |
| 8 | Per-video prediction table |
| 9 | Save results to `outputs/evaluation_results.csv` |

---
## Cell 1 — Imports & Configuration

In [ ]:
import sys
!("{sys.executable}" + " -m pip install opencv-python mediapipe tensorflow seaborn --quiet")

In [ ]:
%pip install mediapipe==0.10.5 --force-reinstall

In [2]:
import json
import os
import warnings
from pathlib import Path

import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
)
import tensorflow as tf
from tensorflow import keras

warnings.filterwarnings('ignore')
np.random.seed(42)

# ── Paths ──────────────────────────────────────────────────────────────────────
ROOT_DIR       = Path('..').resolve()
MODEL_DIR      = ROOT_DIR / 'models'
TEST_DATA_DIR  = ROOT_DIR / 'test_data'
OUTPUT_DIR     = ROOT_DIR / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH     = MODEL_DIR / 'lstm_model.keras'
MODEL_INFO_PATH = MODEL_DIR / 'model_info.json'
RESULTS_CSV    = OUTPUT_DIR / 'evaluation_results.csv'

# ── Feature config (must match preprocessing) ─────────────────────────────────
TARGET_FRAMES  = 60
EXTRACT_FACE   = False
EXTRACT_POSE   = True
EXTRACT_HANDS  = True

HAND_DIM = 21 * 3 * 2 if EXTRACT_HANDS else 0   # 126
POSE_DIM = 25 * 3     if EXTRACT_POSE  else 0   # 75
FACE_DIM = 468 * 3    if EXTRACT_FACE  else 0   # 0
D = HAND_DIM + POSE_DIM + FACE_DIM              # 201
F = D * 2 + 2                                   # 404

print(f'TensorFlow  : {tf.__version__}')
print(f'Root        : {ROOT_DIR}')
print(f'Model       : {MODEL_PATH}')
print(f'Test data   : {TEST_DATA_DIR}')
print(f'Output CSV  : {RESULTS_CSV}')
print(f'Feature dim : D={D}, F={F}')

TensorFlow  : 2.17.0
Root        : C:\Users\Napat\termtemsl
Model       : C:\Users\Napat\termtemsl\models\lstm_model.keras
Test data   : C:\Users\Napat\termtemsl\test_data
Output CSV  : C:\Users\Napat\termtemsl\outputs\evaluation_results.csv
Feature dim : D=201, F=404


---
## Cell 2 — Load Model & Class Metadata

In [3]:
# ── Load trained model ────────────────────────────────────────────────────────
if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f'Model not found: {MODEL_PATH}\n'
        'Run 03_training.ipynb first to produce the trained model.'
    )

model = keras.models.load_model(str(MODEL_PATH))
print('Model loaded successfully.')
model.summary()

# ── Load class metadata ───────────────────────────────────────────────────────
with open(MODEL_INFO_PATH, encoding='utf-8') as f:
    model_info = json.load(f)

CLASS_NAMES   = model_info['class_names']   # ordered list of Thai labels
CLASS_TO_IDX  = model_info['class_to_idx']
NUM_CLASSES   = model_info['num_classes']

print(f'\nClasses ({NUM_CLASSES} total):')
for i, name in enumerate(CLASS_NAMES):
    print(f'  {i:2d}: {name}')

Model loaded successfully.


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ masking (Masking)               │ (None, 60, 404)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 60, 128)        │       272,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 60, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 25)             │         1,625 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 984,781 (3.76 MB)

 Trainable params: 328,217 (1.25 MB)

 Non-trainable params: 128 (512.00 B)

 Optimizer params: 656,436 (2.50 MB)


Classes (25 total):
   0: กิน
   1: ขอบคุณ
   2: ขอโทษ
   3: ดี
   4: ดื่ม
   5: ที่ไหน
   6: นอน
   7: บ้าน
   8: มาก
   9: รถ
  10: ร้อน
  11: สวัสดี
  12: หยุด
  13: อะไร
  14: เงิน
  15: เดิน
  16: เท่าไหร่
  17: เมื่อไหร่
  18: เย็น
  19: โทรศัพท์
  20: โรงพยาบาล
  21: ใคร
  22: ใช่
  23: ไม่
  24: ไม่ดี


---
## Cell 3 — Feature-Extraction Pipeline

Replicates the exact pipeline from `02_preprocessing.ipynb`:

```
video.mp4
  └─ extract_keypoints()   →  (n_frames, D=201)
       └─ compute_features() →  (n_frames, F=404)
            └─ normalize_sequence() → (60, 404)
```

In [4]:
mp_holistic = mp.solutions.holistic


def extract_keypoints(video_path: Path) -> np.ndarray:
    """Run MediaPipe Holistic on every frame; return (n_frames, D) float32."""
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise OSError(f'Cannot open video: {video_path}')

    frames_kp = []
    with mp_holistic.Holistic(
        static_image_mode=False,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5,
    ) as holistic:
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            rgb     = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = holistic.process(rgb)
            parts   = []

            if EXTRACT_HANDS:
                lh = (np.array([[lm.x, lm.y, lm.z]
                                for lm in results.left_hand_landmarks.landmark],
                               dtype=np.float32).flatten()
                      if results.left_hand_landmarks
                      else np.zeros(63, dtype=np.float32))
                rh = (np.array([[lm.x, lm.y, lm.z]
                                for lm in results.right_hand_landmarks.landmark],
                               dtype=np.float32).flatten()
                      if results.right_hand_landmarks
                      else np.zeros(63, dtype=np.float32))
                parts += [lh, rh]

            if EXTRACT_POSE:
                pose = (np.array([[lm.x, lm.y, lm.z]
                                  for lm in results.pose_landmarks.landmark[:25]],
                                 dtype=np.float32).flatten()
                        if results.pose_landmarks
                        else np.zeros(75, dtype=np.float32))
                parts.append(pose)

            if EXTRACT_FACE:
                face = (np.array([[lm.x, lm.y, lm.z]
                                  for lm in results.face_landmarks.landmark],
                                 dtype=np.float32).flatten()
                        if results.face_landmarks
                        else np.zeros(1404, dtype=np.float32))
                parts.append(face)

            frames_kp.append(np.concatenate(parts))

    cap.release()
    return (np.array(frames_kp, dtype=np.float32)
            if frames_kp
            else np.zeros((1, D), dtype=np.float32))


def compute_features(keypoints: np.ndarray) -> np.ndarray:
    """Append velocity + hand stats → (n_frames, F=404)."""
    n_frames, _ = keypoints.shape
    velocity = np.zeros_like(keypoints)
    if n_frames > 1:
        velocity[1:] = np.diff(keypoints, axis=0)

    if EXTRACT_HANDS:
        left_wrist  = keypoints[:, 0:3]
        right_wrist = keypoints[:, 63:66]
        hand_dist   = np.linalg.norm(left_wrist - right_wrist, axis=1, keepdims=True)
        dom_speed   = np.linalg.norm(velocity[:, 0:3], axis=1, keepdims=True)
    else:
        hand_dist = np.zeros((n_frames, 1), dtype=np.float32)
        dom_speed = np.zeros((n_frames, 1), dtype=np.float32)

    return np.concatenate([keypoints, velocity, hand_dist, dom_speed],
                          axis=1).astype(np.float32)


def normalize_sequence(seq: np.ndarray) -> np.ndarray:
    """Resample to TARGET_FRAMES then z-score normalize per feature."""
    n_frames, n_feats = seq.shape
    if n_frames == TARGET_FRAMES:
        resampled = seq.copy()
    elif n_frames < TARGET_FRAMES:
        x_old     = np.linspace(0.0, 1.0, n_frames)
        x_new     = np.linspace(0.0, 1.0, TARGET_FRAMES)
        resampled = np.stack(
            [np.interp(x_new, x_old, seq[:, i]) for i in range(n_feats)], axis=1
        )
    else:
        indices   = np.round(np.linspace(0, n_frames - 1, TARGET_FRAMES)).astype(int)
        resampled = seq[indices]

    eps  = 1e-8
    mean = resampled.mean(axis=0, keepdims=True)
    std  = resampled.std(axis=0,  keepdims=True)
    return ((resampled - mean) / (std + eps)).astype(np.float32)


def video_to_features(video_path: Path) -> np.ndarray:
    """Full pipeline: video → (1, TARGET_FRAMES, F) ready for model.predict."""
    kp   = extract_keypoints(video_path)
    feat = compute_features(kp)
    norm = normalize_sequence(feat)
    return norm[np.newaxis, ...]  # (1, 60, 404)


print('Feature-extraction functions defined.')
print(f'  D (raw keypoints) = {D}')
print(f'  F (features)      = {F}')
print(f'  TARGET_FRAMES     = {TARGET_FRAMES}')

Feature-extraction functions defined.
  D (raw keypoints) = 201
  F (features)      = 404
  TARGET_FRAMES     = 60


---
## Cell 4 — Scan `test_data/` and Run Predictions

Filename convention: `{number}_{thai_label}.mp4`  
The true label is extracted by splitting on the **first** `_` and taking the right side.

In [5]:
video_files = sorted(TEST_DATA_DIR.glob('*.mp4'))

if not video_files:
    raise FileNotFoundError(
        f'No .mp4 files found in {TEST_DATA_DIR}. '
        'Please add test videos with filenames like "01_สวัสดี.mp4".'
    )

print(f'Found {len(video_files)} test videos in {TEST_DATA_DIR}\n')

records = []

for video_path in video_files:
    filename  = video_path.name
    stem      = video_path.stem          # e.g. '01_สวัสดี'

    # Parse true label from filename: split on FIRST '_'
    parts     = stem.split('_', 1)
    true_label = parts[1] if len(parts) == 2 else stem

    # ── Feature extraction ────────────────────────────────────────────────────
    try:
        features = video_to_features(video_path)   # (1, 60, 404)
        proba    = model.predict(features, verbose=0)[0]  # (NUM_CLASSES,)
        pred_idx = int(np.argmax(proba))
        pred_label  = CLASS_NAMES[pred_idx]
        confidence  = float(proba[pred_idx])
        top3        = np.argsort(proba)[::-1][:3]
        top3_labels = [(CLASS_NAMES[i], float(proba[i])) for i in top3]
        error_msg   = ''

    except Exception as exc:
        pred_label  = 'ERROR'
        confidence  = 0.0
        top3_labels = []
        error_msg   = str(exc)
        print(f'  [ERROR] {filename}: {exc}')

    is_correct = (pred_label == true_label)

    records.append({
        'filename'       : filename,
        'true_label'     : true_label,
        'predicted_label': pred_label,
        'confidence_pct' : round(confidence * 100, 2),
        'correct'        : is_correct,
        'top3'           : str(top3_labels),
        'error'          : error_msg,
    })

    mark = '✓' if is_correct else '✗'
    print(
        f'  {mark}  {filename:<32}  '
        f'true={true_label:<14}  '
        f'pred={pred_label:<14}  '
        f'{confidence*100:5.1f}%'
    )

df = pd.DataFrame(records)
valid_df = df[df['predicted_label'] != 'ERROR'].copy()

print(f'\nProcessed : {len(df)} videos  |  Errors : {(df["predicted_label"]=="ERROR").sum()}')

Found 50 test videos in C:\Users\Napat\termtemsl\test_data

  ✗  01_กิน.mp4                        true=กิน             pred=สวัสดี          100.0%
  ✗  01_ขอบคุณ.mp4                     true=ขอบคุณ          pred=ขอโทษ            98.8%
  ✗  01_ขอโทษ.mp4                      true=ขอโทษ           pred=มาก             100.0%
  ✗  01_ดี.mp4                         true=ดี              pred=อะไร             80.8%
  ✗  01_ดื่ม.mp4                       true=ดื่ม            pred=นอน              88.5%
  ✗  01_ที่ไหน.mp4                     true=ที่ไหน          pred=ดี               95.5%
  ✗  01_นอน.mp4                        true=นอน             pred=ใคร             100.0%
  ✗  01_บ้าน.mp4                       true=บ้าน            pred=ใคร              51.5%
  ✓  01_มาก.mp4                        true=มาก             pred=มาก             100.0%
  ✓  01_รถ.mp4                         true=รถ              pred=รถ              100.0%
  ✓  01_ร้อน.mp4                       true=ร้อน            

KeyboardInterrupt: 

---
## Cell 5 — Overall Accuracy & Classification Report

In [ ]:
y_true = valid_df['true_label'].tolist()
y_pred = valid_df['predicted_label'].tolist()

overall_accuracy = accuracy_score(y_true, y_pred)
n_correct        = int(overall_accuracy * len(valid_df))

print('=' * 62)
print(f'  OVERALL ACCURACY : {overall_accuracy*100:.1f}%'
      f'  ({n_correct} / {len(valid_df)} videos correct)')
print('=' * 62)

# Present classes actually appearing in test data
present_classes = sorted(set(y_true) | set(y_pred))

print('\nClassification Report')
print('─' * 62)
print(classification_report(
    y_true, y_pred,
    labels=present_classes,
    zero_division=0,
))

---
## Cell 6 — Confusion Matrix Heatmap

In [ ]:
labels_ordered = sorted(set(y_true))   # only classes present in test set
cm             = confusion_matrix(y_true, y_pred, labels=labels_ordered)
cm_norm        = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(min=1)

n_classes_shown = len(labels_ordered)
fig_size        = max(10, n_classes_shown * 0.6)

fig, ax = plt.subplots(figsize=(fig_size, fig_size * 0.85))

sns.heatmap(
    cm_norm,
    annot=True,
    fmt='.2f',
    xticklabels=labels_ordered,
    yticklabels=labels_ordered,
    cmap='Blues',
    vmin=0, vmax=1,
    linewidths=0.4,
    ax=ax,
    cbar_kws={'label': 'Proportion'},
)

ax.set_title(
    f'Normalized Confusion Matrix — Test Set\n'
    f'Accuracy: {overall_accuracy*100:.1f}%  ({n_correct}/{len(valid_df)})',
    fontsize=13, fontweight='bold', pad=14,
)
ax.set_xlabel('Predicted Label', fontsize=11)
ax.set_ylabel('True Label', fontsize=11)
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()

cm_save = OUTPUT_DIR / 'confusion_matrix_test.png'
plt.savefig(cm_save, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {cm_save}')

---
## Cell 7 — Confidence Score Distribution

Two panels:
- **Left**: histogram of confidence % split by correct / incorrect predictions  
- **Right**: per-class mean confidence, sorted ascending (highlights uncertain classes)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('Prediction Confidence Analysis', fontsize=14, fontweight='bold')

# ── Left: histogram by outcome ────────────────────────────────────────────────
ax = axes[0]
correct_conf   = valid_df.loc[valid_df['correct'],  'confidence_pct']
incorrect_conf = valid_df.loc[~valid_df['correct'], 'confidence_pct']

bins = np.linspace(0, 100, 21)
ax.hist(correct_conf,   bins=bins, alpha=0.7, color='#2ecc71', label=f'Correct   (n={len(correct_conf)})',   edgecolor='white')
ax.hist(incorrect_conf, bins=bins, alpha=0.7, color='#e74c3c', label=f'Incorrect (n={len(incorrect_conf)})', edgecolor='white')
ax.axvline(correct_conf.mean() if len(correct_conf) > 0 else 0,
           color='#27ae60', linestyle='--', linewidth=1.5,
           label=f'Mean correct: {correct_conf.mean():.1f}%' if len(correct_conf) > 0 else '')
if len(incorrect_conf) > 0:
    ax.axvline(incorrect_conf.mean(), color='#c0392b', linestyle='--', linewidth=1.5,
               label=f'Mean incorrect: {incorrect_conf.mean():.1f}%')

ax.set_xlabel('Confidence (%)', fontsize=11)
ax.set_ylabel('Number of Videos', fontsize=11)
ax.set_title('Confidence Distribution by Outcome')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.25)

# ── Right: per-class mean confidence (bar chart, sorted) ──────────────────────
ax = axes[1]
per_class_conf = (
    valid_df.groupby('true_label')['confidence_pct']
    .mean()
    .sort_values()
)
per_class_acc = valid_df.groupby('true_label')['correct'].mean() * 100

colors = ['#2ecc71' if per_class_acc.get(cls, 0) == 100 else '#e67e22'
          if per_class_acc.get(cls, 0) >= 50 else '#e74c3c'
          for cls in per_class_conf.index]

bars = ax.barh(per_class_conf.index, per_class_conf.values, color=colors, edgecolor='white')
ax.axvline(80, color='grey', linestyle=':', linewidth=1, label='80% threshold')
ax.set_xlabel('Mean Confidence (%)', fontsize=11)
ax.set_title('Mean Confidence per True Class\n(green=100% acc, orange=partial, red=0%)')
ax.set_xlim(0, 105)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.25, axis='x')

# Annotate bars with exact value
for bar, val in zip(bars, per_class_conf.values):
    ax.text(val + 0.5, bar.get_y() + bar.get_height() / 2,
            f'{val:.0f}%', va='center', fontsize=8)

plt.tight_layout()
conf_save = OUTPUT_DIR / 'confidence_analysis.png'
plt.savefig(conf_save, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {conf_save}')

---
## Cell 8 — Per-Video Prediction Results Table

Full breakdown for every test video — useful for spotting systematic errors.

In [ ]:
# ── Styled display table ──────────────────────────────────────────────────────
display_df = valid_df[[
    'filename', 'true_label', 'predicted_label', 'confidence_pct', 'correct'
]].copy()
display_df.columns = ['Filename', 'True Label', 'Predicted Label', 'Confidence (%)', 'Correct']
display_df['Correct'] = display_df['Correct'].map({True: '✓', False: '✗'})
display_df = display_df.sort_values(['True Label', 'Filename']).reset_index(drop=True)

def highlight_row(row):
    if row['Correct'] == '✓':
        return ['background-color: #d5f5e3'] * len(row)
    return ['background-color: #fadbd8'] * len(row)

styled = (
    display_df.style
    .apply(highlight_row, axis=1)
    .format({'Confidence (%)': '{:.1f}%'})
    .set_caption(
        f'Per-Video Evaluation Results  —  '
        f'Accuracy: {overall_accuracy*100:.1f}%  '
        f'({n_correct}/{len(valid_df)})'
    )
)

from IPython.display import display
display(styled)

# ── Error videos (if any) ─────────────────────────────────────────────────────
error_df = df[df['predicted_label'] == 'ERROR']
if not error_df.empty:
    print(f'\n{len(error_df)} video(s) could not be processed:')
    for _, row in error_df.iterrows():
        print(f'  {row["filename"]}  —  {row["error"]}')

### Per-Class Accuracy Summary

In [ ]:
per_class_summary = (
    valid_df.groupby('true_label')
    .agg(
        n_videos=('correct', 'count'),
        n_correct=('correct', 'sum'),
        accuracy_pct=('correct', lambda x: round(x.mean() * 100, 1)),
        mean_confidence=('confidence_pct', lambda x: round(x.mean(), 1)),
    )
    .reset_index()
    .sort_values('accuracy_pct')
    .rename(columns={
        'true_label'      : 'Class',
        'n_videos'        : 'Videos',
        'n_correct'       : 'Correct',
        'accuracy_pct'    : 'Accuracy (%)',
        'mean_confidence' : 'Mean Confidence (%)',
    })
)

def color_accuracy(val):
    if val == 100:
        return 'color: #27ae60; font-weight: bold'
    if val >= 50:
        return 'color: #e67e22; font-weight: bold'
    return 'color: #e74c3c; font-weight: bold'

display(
    per_class_summary.style
    .applymap(color_accuracy, subset=['Accuracy (%)'])
    .format({'Accuracy (%)': '{:.1f}%', 'Mean Confidence (%)': '{:.1f}%'})
    .set_caption('Per-Class Accuracy Summary')
    .hide(axis='index')
)

print(f'\nTop-3 hardest classes (lowest accuracy):')
for _, row in per_class_summary.head(3).iterrows():
    print(f'  {row["Class"]:<20}  {row["Accuracy (%)"]:.0f}%  '
          f'({int(row["Correct"])}/{int(row["Videos"])})')

---
## Cell 9 — Save Evaluation Results to CSV

Writes `outputs/evaluation_results.csv` with one row per video for downstream analysis.

In [ ]:
export_df = df[[
    'filename', 'true_label', 'predicted_label', 'confidence_pct', 'correct', 'error'
]].copy()

# Add summary metadata row at end as a comment-like record
export_df.to_csv(RESULTS_CSV, index=False, encoding='utf-8-sig')  # utf-8-sig for Excel compat

# Also write a small summary JSON
summary = {
    'total_videos'      : int(len(df)),
    'valid_videos'      : int(len(valid_df)),
    'error_videos'      : int((df['predicted_label'] == 'ERROR').sum()),
    'correct'           : int(n_correct),
    'overall_accuracy'  : round(float(overall_accuracy), 4),
    'mean_confidence_pct': round(float(valid_df['confidence_pct'].mean()), 2),
    'per_class_accuracy': {
        row['Class']: row['Accuracy (%)']
        for _, row in per_class_summary.iterrows()
    },
}
summary_path = OUTPUT_DIR / 'evaluation_summary.json'
with open(summary_path, 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print(f'Results CSV  saved → {RESULTS_CSV}')
print(f'Summary JSON saved → {summary_path}')
print()
print('─' * 50)
print(f'  Total videos         : {summary["total_videos"]}')
print(f'  Valid (no error)     : {summary["valid_videos"]}')
print(f'  Correct predictions  : {summary["correct"]}')
print(f'  Overall accuracy     : {summary["overall_accuracy"]*100:.1f}%')
print(f'  Mean confidence      : {summary["mean_confidence_pct"]:.1f}%')
print('─' * 50)
print()
print('Evaluation complete.')